# Temperature and salinity of the crossing eddies

Profiles (0 to 1000 dbar) inside the speed contour of a target eddy on the profile day, compared with background profiles on each side of the daily Gulf Stream axis. Profiles come from Argo floats; background profiles are calm (|Ro| < 0.1 on the nearest SWOT day) and outside every tracked eddy contour. Temperature is Conservative Temperature and salinity is Absolute Salinity (TEOS-10, `gsw`).

- Lateral entrainment along isopycnals moves an eddy's T/S curve along the $\sigma_0$ lines toward the surrounding water. Surface cooling, evaporation, and rain act on the mixed layer only and move it across the $\sigma_0$ lines.

In [ ]:
from pathlib import Path
from typing import cast
import datetime as dt
import re
import sys

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import gsw
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from cartopy.mpl.geoaxes import GeoAxes
from IPython.display import display
from matplotlib import patheffects
from matplotlib.axes import Axes
from matplotlib.cm import ScalarMappable
from matplotlib.colors import LinearSegmentedColormap, LogNorm, Normalize
from matplotlib.figure import Figure
from matplotlib.lines import Line2D
from matplotlib.path import Path as MplPath
from matplotlib.ticker import MaxNLocator
from scipy.integrate import trapezoid

PROJECT_ROOT = Path('/Users/jerry/school/research/eddy-tracking')
sys.path.insert(0, str(PROJECT_ROOT))
from eddy_tracking.config import load_analysis_config, resolve_output_dir
from eddy_tracking.preprocess.streamline import compute_signed_distance_grid_km, compute_signed_distance_km, index_centerlines_by_date
from eddy_tracking.preprocess.swot import compute_calm_mask_on_pace, find_nearest_swot_file, index_swot_files_by_date, load_rossby_field
from eddy_tracking.preprocess.tracks import is_in_any_contour, load_date_eddy_index, load_track_observations, mask_pixels_inside_contour
from eddy_tracking.utils.analysis_period import restrict_to_analysis_period
from eddy_tracking.utils.composite_regions import locate_interior
from eddy_tracking.utils.phase_position import add_phase_position, measure_distance_past_axis

EXPERIMENT = 'gulf_stream_20240305_20260531'
NEAR_AXIS_KM = 150
MAX_PRESSURE = 1000
# Background profiles count as the same water when they lie within this many degrees of latitude and days of the eddy profile
REFERENCE_LAT_DEG = 1
REFERENCE_DAYS = 60
DATA_DIR = PROJECT_ROOT / 'data' / EXPERIMENT
cfg = load_analysis_config(EXPERIMENT)
target_classes = {'cyclone': 'NS', 'anticyclone': 'SN'}
polarity_colors = {'cyclone': '#2166ac', 'anticyclone': '#b2182b'}
side_colors = {'north': '#4393c3', 'south': '#d6604d'}
identity_columns = ['polarity', 'track_id']

plt.rcParams.update({
    'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'mathtext.fontset': 'custom', 'mathtext.rm': 'Arial', 'mathtext.it': 'Arial:italic', 'mathtext.bf': 'Arial:bold',
    # matplotlib 3.11 warns about its default calligraphic face, Apple Chancery, even though no label uses it
    'mathtext.cal': 'Arial',
    'font.size': 8, 'axes.titlesize': 8, 'axes.labelsize': 8, 'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 8,
    'axes.linewidth': 0.6, 'xtick.major.width': 0.6, 'ytick.major.width': 0.6, 'xtick.major.size': 2.5, 'ytick.major.size': 2.5,
    'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False,
    'figure.dpi': 150, 'savefig.dpi': 300,
})

eddy_tracks = restrict_to_analysis_period(pd.read_parquet(DATA_DIR / 'silver/gulf_stream/eddy_movement.parquet'), cfg)
target_tracks = eddy_tracks.loc[eddy_tracks['movement'].eq(cast(pd.Series, eddy_tracks['polarity']).map(target_classes))]
track_dir = resolve_output_dir(EXPERIMENT, 'eddy_track')
observations = load_track_observations(track_dir)
observations_by_date = dict(tuple(observations.groupby('date')))
date_index = load_date_eddy_index(track_dir)
centerlines = index_centerlines_by_date(pd.read_parquet(DATA_DIR / 'silver/gulf_stream/streamline.parquet'))
swot_files = index_swot_files_by_date(DATA_DIR / 'bronze/swot_l4_open_ocean')
crossing_observations = measure_distance_past_axis(track_dir, resolve_output_dir(EXPERIMENT, 'gulf_stream'), target_tracks)

# The download kept only good or probably good position, time, pressure and temperature, preferring adjusted values, and left salinity NaN where it failed QC, so some floats give temperature-only profiles
argo = xr.open_dataset(DATA_DIR / 'bronze/argo/core_profiles.nc')
pressure = argo['PRES'].to_numpy().astype(float)
profiles = pd.DataFrame({
    'float_id': argo['PLATFORM_NUMBER'].to_numpy().astype(str),
    'lon': argo['LONGITUDE'].to_numpy().astype(float),
    'lat': argo['LATITUDE'].to_numpy().astype(float),
    'date': argo['TIME'].dt.floor('D').to_numpy(),
})
lon_column = profiles['lon'].to_numpy()[:, None]
lat_column = profiles['lat'].to_numpy()[:, None]
practical_salinity = argo['PSAL'].to_numpy().astype(float)
# Float 4902109 reports salinity near 0 on every level from November 2025 on, a failed conductivity sensor that passed the quality flags
practical_salinity[practical_salinity < 20] = np.nan
absolute_salinity = gsw.SA_from_SP(practical_salinity, pressure, lon_column, lat_column)
# Levels without salinity take 36 g/kg, the median of the levels with it; on those levels that keeps Conservative Temperature within 0.04 °C of the measured-salinity value 99% of the time
conservative_temperature = gsw.CT_from_t(np.where(np.isfinite(absolute_salinity), absolute_salinity, 36.0), argo['TEMP'].to_numpy().astype(float), pressure)
sigma0 = gsw.sigma0(absolute_salinity, conservative_temperature)
# Depth in meters for plot axes; 1000 dbar is about 990 m at these latitudes
depth = -gsw.z_from_p(pressure, lat_column)
valid = np.isfinite(pressure) & np.isfinite(sigma0) & (pressure <= MAX_PRESSURE)
# Levels usable for temperature alone, with or without salinity
valid_temperature = np.isfinite(pressure) & np.isfinite(conservative_temperature) & (pressure <= MAX_PRESSURE)

## Data availability

In [ ]:
# Each profile is matched to the eddy contours of its own day; a profile inside a target eddy keeps that eddy's identity and its distance from the center in speed radii
matches = []
for profile_index, lon, lat, day in zip(profiles.index, profiles['lon'], profiles['lat'], profiles['date']):
    day_observations = observations_by_date[day]
    nearby = day_observations.loc[day_observations['center_lon'].sub(lon).abs().lt(3) & day_observations['center_lat'].sub(lat).abs().lt(3)]
    for polarity, track_id, contour_lon, contour_lat, center_lon, center_lat, radius_km in zip(nearby['polarity'], nearby['track_id'], nearby['contour_lon'], nearby['contour_lat'], nearby['center_lon'], nearby['center_lat'], nearby['radius_km']):
        if MplPath(np.column_stack([contour_lon, contour_lat])).contains_point((lon, lat)):
            distance_km = np.hypot((lon - center_lon) * 111.32 * np.cos(np.radians(lat)), (lat - center_lat) * 111.32)
            matches.append({'profile': profile_index, 'polarity': polarity, 'track_id': track_id, 'radius_fraction': distance_km / radius_km})
matches = pd.DataFrame(matches)
profiles['in_eddy'] = profiles.index.isin(matches['profile'])

# Positive north of the daily axis
profiles['axis_km'] = [
    compute_signed_distance_km(centerlines[day.date()].lon, centerlines[day.date()].lat, lon, lat)[0]
    for day, lon, lat in zip(profiles['date'], profiles['lon'], profiles['lat'])
]
# Background water is calm (|Ro| < 0.1 on the nearest SWOT day) and outside every tracked eddy contour, the rule of the pipeline background stage
abs_rossby = pd.Series(np.nan, index=profiles.index)
for day, day_profiles in profiles.groupby('date'):
    swot_path = find_nearest_swot_file(swot_files, cast(pd.Timestamp, day).date())
    if swot_path is None:
        continue
    swot_lon, swot_lat, rossby_number = load_rossby_field(swot_path)
    field = xr.DataArray(np.abs(rossby_number), coords={'latitude': swot_lat, 'longitude': swot_lon}, dims=['latitude', 'longitude'])
    abs_rossby[day_profiles.index] = field.interp(latitude=xr.DataArray(day_profiles['lat'].to_numpy(), dims='point'), longitude=xr.DataArray(day_profiles['lon'].to_numpy(), dims='point')).to_numpy()
profiles['side'] = np.where(profiles['axis_km'] > 0, 'north', 'south')
# A profile whose every level failed the checks, such as the masked salinity of float 4902109, is no reference
profiles['background'] = ~profiles['in_eddy'] & abs_rossby.lt(0.1) & valid.any(axis=1)
profiles['background_temperature'] = ~profiles['in_eddy'] & abs_rossby.lt(0.1) & valid_temperature.any(axis=1)

interior = matches.merge(target_tracks[identity_columns], on=identity_columns).join(profiles, on='profile')
interior = add_phase_position(interior, target_tracks, crossing_observations, NEAR_AXIS_KM)
interior['days_on_new_side'] = (interior['date'] - interior['departure_date']).dt.days
# The crossing is the first day the center is past the daily axis; some eddies wobble back across it later
crossing_dates = cast(pd.Series, crossing_observations.loc[crossing_observations['across_km'].gt(0)].groupby(identity_columns)['date'].min())
interior['days_from_crossing'] = [(date - crossing_dates[polarity, track_id]).days for date, polarity, track_id in zip(interior['date'], interior['polarity'], interior['track_id'])]
# Temperature-only profiles (salinity failed QC) go only into interior_temperature, for the temperature-only figures; interior keeps the profiles with salinity
interior_temperature = interior
interior = interior.loc[valid.any(axis=1)[interior['profile'].to_numpy()]]
# Keep only target eddies with at least 4 Argo profiles with salinity inside their contour, before and after the crossing together; every later cell reads this filtered interior
interior = interior.loc[interior.groupby(identity_columns)['profile'].transform('size').ge(4)]
interior_temperature = interior_temperature.merge(interior[identity_columns].drop_duplicates(), on=identity_columns)

profile_groups = {
    'All Argo profiles': np.ones(len(profiles), dtype=bool),
    'Inside any tracked eddy': profiles['in_eddy'].to_numpy(),
    'Inside a target eddy with 4+ profiles': profiles.index.isin(interior['profile']),
    'Temperature only, inside those eddies': profiles.index.isin(interior_temperature['profile']) & ~valid.any(axis=1),
    'North-side background': (profiles['background'] & profiles['side'].eq('north')).to_numpy(),
    'South-side background': (profiles['background'] & profiles['side'].eq('south')).to_numpy(),
    'Temperature-only background': (profiles['background_temperature'] & ~profiles['background']).to_numpy(),
}
display(pd.DataFrame({name: {'profiles': selected.sum(), 'floats': profiles.loc[selected, 'float_id'].nunique()} for name, selected in profile_groups.items()}).T)
eddy_counts = cast(pd.DataFrame, interior.groupby(identity_columns).agg(
    before_crossing=('days_from_crossing', lambda days: (days < 0).sum()), after_crossing=('days_from_crossing', lambda days: (days >= 0).sum()),
    floats=('float_id', 'nunique'), first_day=('days_from_crossing', 'min'), last_day=('days_from_crossing', 'max'),
))
eddy_counts['temperature_only'] = interior_temperature.groupby(identity_columns).size() - eddy_counts['before_crossing'] - eddy_counts['after_crossing']
display(eddy_counts)

# Cyclones 284 and 379 stay out of every figure below for now: their profiles look odd and are still under investigation. The tables above still list them
interior = interior.loc[~(interior['polarity'].eq('cyclone') & interior['track_id'].isin([284, 379]))]
interior_temperature = interior_temperature.merge(interior[identity_columns].drop_duplicates(), on=identity_columns)

## Seasonal baseline

Temperature follows the seasons, so two profiles taken months apart differ even when they hold the same water. Wherever a comparison spans seasons at a fixed depth, temperature is shown as an anomaly: the measured value minus the seasonal baseline of the region for that depth and day of the year. The baseline is the World Ocean Atlas 2023 monthly climatology (1991 to 2020 means on a 1° grid), converted to Conservative Temperature, averaged over the open-ocean cells of the region, and fitted at each depth with an annual and a semiannual harmonic so that it changes smoothly from one month to the next.

- One baseline serves the whole region, so values measured on the same day keep their differences, including the step across the Gulf Stream. A baseline taken at each profile's own position would remove that step, because the climatology holds it too.
- North of the axis the seasons swing about twice as far as the regional baseline, so north-side anomalies near the surface keep part of the seasonal cycle.
- Plots against density keep the measured temperature. Summer warming makes new, lighter water at the surface and leaves the deeper density surfaces nearly unchanged.

In [ ]:
# The regional baseline: World Ocean Atlas 2023 monthly means over 1991 to 2020 as Conservative Temperature, averaged over the cells that reach 1000 m, so every depth averages the same open-ocean cells, where the Argo floats drift
woa = xr.open_dataset(DATA_DIR / 'bronze/woa23/woa23_monthly.nc')
woa_depth = woa['depth'].to_numpy()
woa_lat = woa['lat'].to_numpy()[:, None]
woa_lon = woa['lon'].to_numpy()[None, :]
woa_pressure = gsw.p_from_z(-woa_depth[:, None, None], woa_lat)
woa_temperature = gsw.CT_from_t(gsw.SA_from_SP(woa['s_an'].to_numpy(), woa_pressure, woa_lon, woa_lat), woa['t_an'].to_numpy(), woa_pressure)  # (month, depth, lat, lon)
weights = np.cos(np.radians(woa_lat)) * np.isfinite(woa_temperature[:, woa_depth == 1000]).all(axis=(0, 1))
regional_temperature = np.nansum(woa_temperature * weights, axis=(2, 3)) / (np.isfinite(woa_temperature) * weights).sum(axis=(2, 3))  # (month, depth)


def compute_harmonics(day_of_year: np.ndarray) -> np.ndarray:
    phase = 2 * np.pi * day_of_year / 365.25
    return np.stack([np.ones_like(phase), np.cos(phase), np.sin(phase), np.cos(2 * phase), np.sin(2 * phase)], axis=-1)


# An annual and a semiannual harmonic through the 12 monthly means at each depth, each month placed on its 15th day
month_days = pd.to_datetime([f'2025-{month:02d}-15' for month in range(1, 13)]).dayofyear.to_numpy().astype(float)
seasonal_coefficients = np.linalg.lstsq(compute_harmonics(month_days), regional_temperature, rcond=None)[0]  # (5, depth)


def estimate_seasonal_baseline(day_of_year: int, depth_m: np.ndarray) -> np.ndarray:
    return np.interp(depth_m, woa_depth, compute_harmonics(np.array(float(day_of_year))) @ seasonal_coefficients, left=np.nan, right=np.nan)


# Each Argo level minus the baseline at its depth on the profile's day of the year
temperature_anomaly = conservative_temperature - np.array([estimate_seasonal_baseline(day, level_depth) for day, level_depth in zip(profiles['date'].dt.dayofyear, depth)])

# The baseline at a few depths: dots are the regional monthly means, lines the fitted harmonics
BASELINE_DEPTHS = (0, 50, 100, 200)
fig, ax = cast(tuple[Figure, Axes], plt.subplots(figsize=(3.35, 2.6), layout='constrained'))
year_days = np.arange(1, 366, dtype=float)
for color, baseline_depth in zip(('#08519c', '#3182bd', '#6baed6', '#9ecae1'), BASELINE_DEPTHS):
    level = int(np.flatnonzero(woa_depth == baseline_depth)[0])
    ax.plot(year_days, compute_harmonics(year_days) @ seasonal_coefficients[:, level], color=color, linewidth=1.2, label=f'{baseline_depth} m')
    ax.scatter(month_days, regional_temperature[:, level], s=10, color=color, edgecolors='white', linewidths=0.4, zorder=3)
ax.set_xlim(1, 365)
ax.xaxis.set_ticks(month_days.tolist(), labels=[f'{day:%b}'[0] for day in pd.date_range('2025-01-01', periods=12, freq='MS')])
ax.set_ylabel('Conservative temperature (°C)')
ax.grid(color='#e5e5e5', linewidth=0.5)
ax.set_axisbelow(True)
ax.legend(loc='upper left', ncol=2, handlelength=1.4, columnspacing=1)
plt.show()

## North and south of the Gulf Stream

In [ ]:
pressure_grid = np.arange(10, MAX_PRESSURE + 1, 10)


# keep picks the levels: a row of valid for temperature with salinity, a row of valid_temperature for temperature alone
def interpolate_to_pressure_grid(profile_index: int, keep: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    levels = np.argsort(pressure[profile_index, keep])
    profile_pressure = pressure[profile_index, keep][levels]
    salinity = np.interp(pressure_grid, profile_pressure, absolute_salinity[profile_index, keep][levels], left=np.nan, right=np.nan)
    temperature = np.interp(pressure_grid, profile_pressure, conservative_temperature[profile_index, keep][levels], left=np.nan, right=np.nan)
    anomaly = np.interp(pressure_grid, profile_pressure, temperature_anomaly[profile_index, keep][levels], left=np.nan, right=np.nan)
    return salinity, temperature, anomaly


# The pressure-weighted 0 to 1000 dbar mean of salinity and of the temperature anomaly, for profiles that reach from the top 10 dbar to 950 dbar
column_means = np.full((len(profiles), 2), np.nan)
for index in range(len(profiles)):
    keep = valid[index]
    levels = np.argsort(pressure[index, keep])
    profile_pressure = pressure[index, keep][levels]
    if len(profile_pressure) < 5 or profile_pressure[0] > 10 or profile_pressure[-1] < 950:
        continue
    span = profile_pressure[-1] - profile_pressure[0]
    column_means[index] = [trapezoid(absolute_salinity[index, keep][levels], profile_pressure) / span, trapezoid(temperature_anomaly[index, keep][levels], profile_pressure) / span]
profiles['column_salinity'], profiles['column_temperature_anomaly'] = column_means.T

# Temperature, salinity, and pressure where each profile first reaches each density surface; a surface lighter than the top level (it outcrops) or deeper than 1000 dbar stays NaN
DENSITY_SURFACES = (26.5, 26.8, 27.1)
on_surfaces = np.full((len(profiles), len(DENSITY_SURFACES), 3), np.nan)  # (profile, surface, [temperature, salinity, pressure])
for index in range(len(profiles)):
    keep = valid[index]
    levels = np.argsort(pressure[index, keep])
    profile_density = sigma0[index, keep][levels]
    profile_values = np.column_stack([conservative_temperature[index, keep][levels], absolute_salinity[index, keep][levels], pressure[index, keep][levels]])
    for column, surface in enumerate(DENSITY_SURFACES):
        below = np.flatnonzero(profile_density >= surface)
        if len(below) == 0 or below[0] == 0:
            continue
        lower = below[0]
        weight = (surface - profile_density[lower - 1]) / (profile_density[lower] - profile_density[lower - 1])
        on_surfaces[index, column] = profile_values[lower - 1] + weight * (profile_values[lower] - profile_values[lower - 1])

# Calm, non-eddy Argo profiles anywhere on each side of the daily axis, pooled over the whole record, so each side's spread holds its range of latitudes, and its seasons in the rows that are not temperature anomalies
METRIC_PRESSURES = (10, 100, 300, 600, 900)
side_rows = []
side_names = {}
for side in ('north', 'south'):
    chosen = profiles.loc[profiles['background'] & profiles['side'].eq(side)]
    side_names[side] = f'{side} ({len(chosen)} profiles)'
    grids = [interpolate_to_pressure_grid(index, valid[index]) for index in chosen.index]
    salinity = np.array([salinity for salinity, _, _ in grids])  # (n_profiles, n_pressures)
    anomaly = np.array([anomaly for _, _, anomaly in grids])
    levels = np.isin(pressure_grid, METRIC_PRESSURES)
    chosen_surfaces = on_surfaces[chosen.index]
    values = {
        **{f'Temperature anomaly at {level} dbar (°C)': anomaly[:, column] for level, column in zip(METRIC_PRESSURES, np.flatnonzero(levels))},
        'Temperature anomaly, 0 to 1000 dbar mean (°C)': chosen['column_temperature_anomaly'].to_numpy(),
        **{f'Temperature on σ0 = {surface} kg/m3 (°C)': chosen_surfaces[:, column, 0] for column, surface in enumerate(DENSITY_SURFACES)},
        **{f'Salinity at {level} dbar (g/kg)': salinity[:, column] for level, column in zip(METRIC_PRESSURES, np.flatnonzero(levels))},
        'Salinity, 0 to 1000 dbar mean (g/kg)': chosen['column_salinity'].to_numpy(),
        **{f'Salinity on σ0 = {surface} kg/m3 (g/kg)': chosen_surfaces[:, column, 1] for column, surface in enumerate(DENSITY_SURFACES)},
        **{f'Pressure of σ0 = {surface} kg/m3 (dbar)': chosen_surfaces[:, column, 2] for column, surface in enumerate(DENSITY_SURFACES)},
    }
    for name, column_values in values.items():
        low, median, high = np.nanquantile(column_values, [0.25, 0.5, 0.75])
        side_rows.append({'side': side, 'quantity': name, 'median': median, 'IQR': high - low})
water_masses = cast(pd.DataFrame, pd.DataFrame(side_rows).pivot(index='quantity', columns='side', values=['median', 'IQR']).reindex(list(values)).swaplevel(axis=1)[['north', 'south']])
display(water_masses.rename(columns=side_names, level=0).round(2))

In [ ]:
def locate_background(lon: np.ndarray, lat: np.ndarray, axis_km: np.ndarray, midpoint: pd.Timestamp, contours: list[tuple[np.ndarray, np.ndarray]]) -> dict[str, np.ndarray]:
    # Background water as in the background pipeline stage: calm (|Ro| < 0.1 on the nearest SWOT day; the open-ocean SWOT files leave only about 1° of coast blank, so shelf water farther out stays in) and outside every eddy contour of the composite's 8 days, split by side of the daily axis
    background = compute_calm_mask_on_pace(find_nearest_swot_file(swot_files, midpoint.date()), lon, lat)
    for contour_lon, contour_lat in contours:
        background &= ~locate_interior(lon, lat, contour_lon, contour_lat)
    return {'north': background & (axis_km > 0), 'south': background & (axis_km < 0)}


# MODIS-Aqua 8-day SST and the SMAP 8-day SSS map centered on the composite's middle day, over the background on each side of the middle day's axis, for every composite of the record
satellite_maps = {}
series_rows = []
for sst_path in sorted((DATA_DIR / 'bronze/sst').glob('*.nc')):
    start, end = (pd.Timestamp(part) for part in sst_path.name.split('.')[1].split('_'))
    midpoint = cast(pd.Timestamp, start + pd.Timedelta(days=3))
    sss_path = DATA_DIR / f'bronze/sss/SMAP_L3_SSS_{midpoint:%Y%m%d}_8DAYS_V5_subsetted.nc4'
    if not sss_path.exists():
        continue
    with xr.open_dataset(sst_path) as sst_file, xr.open_dataset(sss_path) as sss_file:
        # Quality 2 is cloud-contaminated and would fake cold water
        fields = {
            'SST': (sst_file['sst'].where(sst_file['qual_sst'] <= 1).to_numpy(), sst_file['lon'].to_numpy(), sst_file['lat'].to_numpy()),
            'SSS': (sss_file['smap_sss'].where(sss_file['land_fraction'] < 0.01).to_numpy(), sss_file['longitude'].to_numpy(), sss_file['latitude'].to_numpy()),
        }
    satellite_maps[midpoint] = fields
    centerline = centerlines[midpoint.date()]
    contours = [(eddy.contour_lon, eddy.contour_lat) for day in pd.date_range(start, end) for eddy in date_index.get(day.date(), [])]
    for variable, (field, lon, lat) in fields.items():
        sides = locate_background(lon, lat, compute_signed_distance_grid_km(centerline.lon, centerline.lat, lon, lat), midpoint, contours)
        for side, inside in sides.items():
            values = field[inside]
            values = values[np.isfinite(values)]
            # The SST record is the anomaly from the regional seasonal baseline, so the seasons do not swamp the difference between the sides
            if variable == 'SST':
                values = values - float(estimate_seasonal_baseline(midpoint.dayofyear, np.array(0.0)))
            low, high = np.quantile(values, [0.25, 0.75])
            series_rows.append({'date': midpoint, 'variable': variable, 'side': side, 'mean': values.mean(), 'q25': low, 'q75': high, 'pixels': values.size})
side_series = pd.DataFrame(series_rows)

# The map day is a spring composite with little cloud, when the surface front is still sharp
MAP_DAY = cast(pd.Timestamp, pd.Timestamp('2025-04-18'))
fig = cast(Figure, plt.figure(figsize=(6.69, 7.2), layout='constrained'))
grid = fig.add_gridspec(3, 2, height_ratios=[1.2, 1, 1])
map_settings = {
    'SST': ('MODIS-Aqua SST', 'RdYlBu_r', (0, 28, 4), 'SST (°C)'),
    'SSS': ('SMAP SSS', 'viridis', (32, 37, 1), 'SSS (psu)'),
}
series_settings = {
    'SST': ('MODIS-Aqua SST anomaly', 'SST anomaly (°C)', (-15, 6, 3)),
    'SSS': ('SMAP SSS', 'SSS (psu)', (31, 37, 1)),
}
centerline = centerlines[MAP_DAY.date()]
for column, (variable, (title, cmap, (low, high, step), unit)) in enumerate(map_settings.items()):
    field, lon, lat = satellite_maps[MAP_DAY][variable]
    map_ax = cast(GeoAxes, fig.add_subplot(grid[0, column], projection=ccrs.PlateCarree()))
    map_ax.set_extent([-81, -56, 29, 44], crs=ccrs.PlateCarree())
    mesh = map_ax.pcolormesh(lon, lat, field, cmap=cmap, norm=Normalize(low, high), shading='nearest', rasterized=True, transform=ccrs.PlateCarree(), zorder=1)
    map_ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='#e9e9e9', zorder=2)
    map_ax.coastlines(resolution='50m', color='#666666', linewidth=0.5, zorder=3)
    map_ax.plot(centerline.lon, centerline.lat, color='black', linewidth=1, transform=ccrs.PlateCarree(), zorder=4)
    gridlines = map_ax.gridlines(draw_labels=True, linewidth=0.5, color='#bbbbbb', alpha=0.6, xlocs=range(-80, -55, 5), ylocs=range(30, 45, 5), zorder=3)
    gridlines.top_labels = False
    gridlines.right_labels = False
    gridlines.xlabel_style = {'size': 8}
    gridlines.ylabel_style = {'size': 8}
    map_ax.set_title(rf'$\bf{{({"ab"[column]})}}$ {title}, 8 days centered on {MAP_DAY:%-d %b %Y}', loc='left')
    colorbar = fig.colorbar(mesh, ax=map_ax, location='bottom', shrink=0.8, aspect=30, pad=0.02, ticks=np.arange(low, high + step, step))
    colorbar.set_label(unit, labelpad=1)
    colorbar.outline.set_linewidth(0.5)  # pyright: ignore[reportCallIssue]

series_axes = [cast(Axes, fig.add_subplot(grid[1, :]))]
series_axes.append(cast(Axes, fig.add_subplot(grid[2, :], sharex=series_axes[0])))
for ax, letter, (variable, (title, unit, (low, high, step))) in zip(series_axes, 'cd', series_settings.items()):
    for side in ('north', 'south'):
        side_values = side_series.loc[side_series['variable'].eq(variable) & side_series['side'].eq(side)]
        ax.fill_between(side_values['date'], side_values['q25'], side_values['q75'], color=side_colors[side], alpha=0.2, linewidth=0)
        ax.plot(side_values['date'], side_values['mean'], color=side_colors[side], linewidth=1.2)
    ax.axvline(MAP_DAY, color='#999999', linewidth=0.6, zorder=1)  # pyright: ignore[reportArgumentType]
    ax.set_ylim(low, high)
    ax.yaxis.set_ticks(np.arange(low, high + step, step).tolist())
    ax.grid(axis='y', color='#e5e5e5', linewidth=0.5)
    ax.set_axisbelow(True)
    ax.set_ylabel(unit)
    ax.set_title(rf'$\bf{{({letter})}}$ {title} of the background on each side of the Gulf Stream', loc='left')
series_axes[0].tick_params(labelbottom=False)
series_axes[1].xaxis.set_major_locator(mdates.MonthLocator(bymonth=(1, 4, 7, 10)))
series_axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
series_axes[1].set_xlim(side_series['date'].min(), side_series['date'].max())  # pyright: ignore[reportArgumentType]
handles = [
    Line2D([], [], color=side_colors['north'], linewidth=1.2, label='North side, mean'),
    Line2D([], [], color=side_colors['south'], linewidth=1.2, label='South side, mean'),
    Line2D([], [], color='#999999', linewidth=6, alpha=0.4, label='Middle half of the pixels'),
    Line2D([], [], color='#999999', linewidth=0.6, label='Day of maps (a) and (b)'),
]
fig.legend(handles=handles, loc='outside lower center', ncol=4, handlelength=1.6, columnspacing=1.5)
plt.show()

In [ ]:
# The endmembers: the background water on each side of the Gulf Stream by month of the year, the median over the 8-day composites centered in that month of the background mean in the satellite record above
surface_series = side_series[['date', 'variable', 'side', 'mean']].astype({'mean': float})
# The record above holds SST as the anomaly, so the baseline goes back on; here the month is on the x axis, so the seasons are part of what the plot shows
is_sst = surface_series['variable'].eq('SST')
surface_series.loc[is_sst, 'mean'] += [float(estimate_seasonal_baseline(day, np.array(0.0))) for day in surface_series.loc[is_sst, 'date'].dt.dayofyear]
surface_series['month'] = surface_series['date'].dt.month
endmembers = surface_series.groupby(['variable', 'side', 'month'])['mean'].median()

fig, axes = cast(tuple[Figure, np.ndarray], plt.subplots(2, 1, figsize=(3.35, 4.6), sharex=True, layout='constrained'))
endmember_settings = {
    'SST': ('MODIS-Aqua SST', 'SST (°C)', (4, 32, 4)),
    'SSS': ('SMAP SSS', 'SSS (psu)', (32, 37, 1)),
}
for ax, letter, (variable, (title, unit, (low, high, step))) in zip(axes, 'ab', endmember_settings.items()):
    for side in ('north', 'south'):
        side_values = surface_series.loc[surface_series['variable'].eq(variable) & surface_series['side'].eq(side)]
        ax.scatter(side_values['date'].dt.dayofyear, side_values['mean'], s=8, color=side_colors[side], alpha=0.45, linewidths=0)
        ax.plot(month_days, endmembers.loc[(variable, side)].reindex(range(1, 13)), color=side_colors[side], linewidth=1.2, marker='o', markersize=3.5, markeredgecolor='white', markeredgewidth=0.4)
    ax.set_ylim(low, high)
    ax.yaxis.set_ticks(np.arange(low, high + step, step).tolist())
    ax.grid(color='#e5e5e5', linewidth=0.5)
    ax.set_axisbelow(True)
    ax.set_ylabel(unit)
    ax.set_title(rf'$\bf{{({letter})}}$ {title} of the background water', loc='left')
axes[1].set_xlim(1, 365)
axes[1].xaxis.set_ticks(month_days.tolist(), labels=[f'{day:%b}'[0] for day in pd.date_range('2025-01-01', periods=12, freq='MS')])
handles = [
    Line2D([], [], color=side_colors['north'], linewidth=1.2, marker='o', markersize=3.5, markeredgecolor='white', markeredgewidth=0.4, label='North side'),
    Line2D([], [], color=side_colors['south'], linewidth=1.2, marker='o', markersize=3.5, markeredgecolor='white', markeredgewidth=0.4, label='South side'),
    Line2D([], [], color='#999999', linestyle='none', marker='o', markersize=3, alpha=0.6, label='One 8-day composite'),
]
fig.legend(handles=handles, loc='outside lower center', ncol=3, handlelength=1.6, columnspacing=1.2)
plt.show()

## Time-depth sections

Temperature anomaly and salinity of each target eddy through time, against depth. The dashed lines are the density surfaces of the water-mass table. Heave moves the isopycnals and the colors up or down together, so a color change along an isopycnal means the water itself changed. Temperature is the anomaly from the seasonal baseline, because each panel compares one depth across seasons. Black triangles mark the profile days, and the fields between them are interpolated linearly in time.

In [ ]:
# Each target eddy's temperature anomaly and salinity through time against depth; between profiles the fields are interpolated linearly in time, profiles from the same day are averaged, and triangles along the top mark the profile days
section_depths = np.arange(0, 991, 5)
anomaly_levels = np.arange(-8, 8.1, 1)
salinity_levels = np.arange(34, 37.01, 0.25)


def interpolate_in_time(profile_days: np.ndarray, profile_values: np.ndarray, section_days: np.ndarray) -> np.ndarray:
    after = np.clip(np.searchsorted(profile_days, section_days), 1, len(profile_days) - 1)
    weight = ((section_days - profile_days[after - 1]) / (profile_days[after] - profile_days[after - 1]))[:, None]
    return (1 - weight) * profile_values[after - 1] + weight * profile_values[after]


# Each field on section_depths for every profile day (same-day profiles averaged) and interpolated to every day between the first and last profile day
def build_section(eddy_profiles: pd.DataFrame, keep_levels: np.ndarray, fields: tuple[np.ndarray, ...]) -> tuple[np.ndarray, np.ndarray, np.ndarray, list[np.ndarray]]:
    on_depths = []
    for profile_index in eddy_profiles['profile']:
        keep = keep_levels[profile_index]
        levels = np.argsort(depth[profile_index, keep])
        # The shallowest level stands for the mixed layer above it
        on_depths.append([np.interp(section_depths, depth[profile_index, keep][levels], values[profile_index, keep][levels], right=np.nan) for values in fields])
    days = eddy_profiles['days_from_crossing'].to_numpy()
    profile_days = np.unique(days)
    profile_grids = np.array([np.array(on_depths)[days == day].mean(axis=0) for day in profile_days])  # (day, field, depth)
    section_days = np.arange(profile_days[0], profile_days[-1] + 1)
    return profile_days, profile_grids, section_days, [interpolate_in_time(profile_days, profile_grids[:, variable], section_days).T for variable in range(len(fields))]


section_eddies = interior[identity_columns].drop_duplicates().sort_values(identity_columns)
fig, axes = cast(tuple[Figure, np.ndarray], plt.subplots(len(section_eddies), 2, figsize=(6.69, 1.9 * len(section_eddies) + 0.3), layout='constrained'))
for row, (polarity, track_id) in enumerate(zip(section_eddies['polarity'], section_eddies['track_id'])):
    in_eddy = interior_temperature['polarity'].eq(polarity) & interior_temperature['track_id'].eq(track_id)
    temperature_days, _, temperature_section_days, (anomaly_section,) = build_section(interior_temperature.loc[in_eddy], valid_temperature, (temperature_anomaly,))
    profile_days, profile_grids, section_days, (salinity_section, density_section) = build_section(interior.loc[interior['polarity'].eq(polarity) & interior['track_id'].eq(track_id)], valid, (absolute_salinity, sigma0))
    # Mixed-layer depth on each profile day: the first depth where density exceeds its 10 m value by 0.03 kg m-3
    mixed_layer_depths = [section_depths[np.argmax(density > density[section_depths == 10] + 0.03)] for density in profile_grids[:, 1]]

    # Temperature-only profiles add temperature days, so the temperature panel can span more days and mark more profiles than the salinity panel; both share its span
    for column, (column_days, marked_days, section, levels, cmap, name) in enumerate(((temperature_section_days, temperature_days, anomaly_section, anomaly_levels, 'RdBu_r', 'temperature anomaly'), (section_days, profile_days, salinity_section, salinity_levels, 'viridis', 'salinity'))):
        ax = cast(Axes, axes[row, column])
        filled = ax.contourf(column_days, section_depths, section, levels=levels, cmap=cmap, extend='both')
        # Black lines, since white ones vanish into the pale middle of the anomaly scale
        ax.contour(section_days, section_depths, density_section, levels=DENSITY_SURFACES, colors='black', linewidths=0.7, linestyles='dashed')
        ax.plot(profile_days, mixed_layer_depths, color='black', linewidth=1)
        if temperature_days[0] < 0 < temperature_days[-1]:
            ax.axvline(0, color='black', linewidth=1, linestyle=(0, (1, 1.5)))
        # Name each isopycnal beside the right edge, at its depth on the last profile day, kept at least 70 m below the label above it so close lines stay readable
        label_depth = -np.inf
        for surface, line_depth in zip(DENSITY_SURFACES, np.interp(DENSITY_SURFACES, density_section[:, -1], section_depths, left=np.nan, right=np.nan)):
            if np.isfinite(line_depth):
                label_depth = max(line_depth, label_depth + 70)
                ax.annotate(f'{surface:g}', (1, label_depth), xycoords=('axes fraction', 'data'), xytext=(2, 0), textcoords='offset points', va='center', fontsize=6, color='#555555')
        ax.scatter(marked_days, np.full(len(marked_days), 1.04), transform=ax.get_xaxis_transform(), marker='v', s=12, color='black', clip_on=False, zorder=4)
        ax.set_xlim(temperature_section_days[0], temperature_section_days[-1])
        ax.set_ylim(1000, 0)
        ax.xaxis.set_major_locator(MaxNLocator(nbins=6, integer=True))
        ax.grid(color='black', linewidth=0.4, alpha=0.25)
        ax.set_title(rf'$\bf{{({"abcdefghijklmnop"[2 * row + column]})}}$ {polarity.capitalize()} {track_id}, {name}', loc='left', color=polarity_colors[polarity], pad=9)
        if row == 0:
            fig.colorbar(filled, ax=axes[:, column], aspect=40, pad=0.02, label='Temperature anomaly (°C)' if column == 0 else 'Absolute salinity (g kg$^{-1}$)')
    cast(Axes, axes[row, 0]).set_ylabel('Depth (m)')
for ax in axes[-1]:
    cast(Axes, ax).set_xlabel('Days from crossing')
handles = [
    Line2D([], [], color='black', linewidth=0.7, linestyle='dashed', label=r'Isopycnals ($\sigma_0$, kg m$^{-3}$)'),
    Line2D([], [], color='black', linewidth=1, label='Mixed-layer depth'),
    Line2D([], [], color='black', linewidth=1, linestyle=(0, (1, 1.5)), label='Crossing'),
    Line2D([], [], color='black', marker='v', markersize=4, linestyle='none', label='Profile day'),
]
fig.legend(handles=handles, loc='outside lower center', ncol=4, handlelength=2.2)
plt.show()

## Eddies with repeated profiles

In [ ]:
# Eddies with profiles on both sides of the crossing, or at least four after it, mostly from floats that stayed inside the eddy, so each panel follows one body of water through time
profile_counts = interior.groupby(identity_columns)['days_from_crossing'].agg(before=lambda days: (days < 0).sum(), after=lambda days: (days >= 0).sum())
case_eddies = cast(pd.DataFrame, profile_counts.loc[((profile_counts['before'] > 0) & (profile_counts['after'] > 0)) | (profile_counts['after'] >= 4)].reset_index()[identity_columns])
# Profiles are colored from yellow (earliest) to dark green (latest) by their days from the crossing
time_cmap = LinearSegmentedColormap.from_list('yellow_green', ['#f0c000', '#6abf3a', '#00441b'])
marked_pressures = np.isin(pressure_grid, (100, 300, 600))
salinity_grid, temperature_grid = np.meshgrid(np.linspace(33, 37.5, 200), np.linspace(0, 32, 200))
density_grid = gsw.sigma0(salinity_grid, temperature_grid)


def draw_isopycnals(ax: Axes, levels: np.ndarray) -> None:
    lines = ax.contour(salinity_grid, temperature_grid, density_grid, levels=levels, colors='#9a9a9a', linewidths=0.5, zorder=1)
    # Each label sits just inside the edge where its isopycnal leaves the axes: the top, else the left, else the bottom
    (left, right), (bottom, top) = ax.get_xlim(), ax.get_ylim()
    inset_temperature = 0.05 * (top - bottom)
    inset_salinity = 0.04 * (right - left)
    column = np.abs(salinity_grid[0] - (left + inset_salinity)).argmin()
    positions = []
    for level in levels:
        top_salinity = float(np.interp(level, gsw.sigma0(salinity_grid[0], top - inset_temperature), salinity_grid[0], left=np.nan, right=np.nan))
        left_temperature = float(np.interp(-level, -density_grid[:, column], temperature_grid[:, 0], left=np.nan, right=np.nan))
        if left + inset_salinity < top_salinity < right:
            positions.append((top_salinity, top - inset_temperature))
        elif bottom + inset_temperature < left_temperature < top:
            positions.append((left + inset_salinity, left_temperature))
        else:
            positions.append((float(np.interp(level, gsw.sigma0(salinity_grid[0], bottom + inset_temperature), salinity_grid[0])), bottom + inset_temperature))
    ax.clabel(lines, fmt='%g', fontsize=7, inline_spacing=3, manual=positions)
    ax.annotate(r'$\sigma_0$ (kg m$^{-3}$)', (0.02, 0.98), xycoords='axes fraction', ha='left', va='top', color='#7a7a7a')


def select_nearby_background(eddy_profiles: pd.DataFrame, background_column: str = 'background') -> np.ndarray:
    nearby = np.zeros(len(profiles), dtype=bool)
    for lat, day in zip(eddy_profiles['lat'], eddy_profiles['date']):
        nearby |= (profiles[background_column] & profiles['lat'].sub(lat).abs().le(REFERENCE_LAT_DEG) & profiles['date'].sub(day).abs().le(pd.Timedelta(days=REFERENCE_DAYS))).to_numpy()
    return nearby


def add_day_colorbar(fig: Figure, ax: Axes, days: pd.Series) -> Normalize:
    day_norm = Normalize(days.min(), days.max())
    colorbar = fig.colorbar(ScalarMappable(cmap=time_cmap, norm=day_norm), ax=ax, fraction=0.04, pad=0.02, aspect=25)
    colorbar.outline.set_linewidth(0.5)  # pyright: ignore[reportCallIssue]
    colorbar.ax.yaxis.set_major_locator(MaxNLocator(nbins=5, integer=True))
    # A black line marks the crossing when the profiles span it
    if days.min() < 0 < days.max():
        colorbar.ax.axhline(0, color='black', linewidth=0.8)
    colorbar.set_label('Days from crossing', labelpad=2)
    return day_norm


# Two columns of panels; an odd count leaves a spare cell for the legend, an even count puts it under the panels
n_rows = (len(case_eddies) + 1) // 2
spare_cell = len(case_eddies) % 2 == 1
fig, axes = cast(tuple[Figure, np.ndarray], plt.subplots(n_rows, 2, figsize=(6.69, 2.97 * n_rows + (0 if spare_cell else 0.5)), layout='constrained', sharex=True, sharey=True, squeeze=False))
for letter, ax, (polarity, track_id) in zip('abcdefghijklmnopqrstuvwxyz', axes.flat, case_eddies.itertuples(index=False)):
    ax = cast(Axes, ax)
    eddy_profiles = interior.loc[interior['polarity'].eq(polarity) & interior['track_id'].eq(track_id)].sort_values('date')
    ax.set_xlim(34, 37.5)
    ax.set_ylim(4, 30)
    draw_isopycnals(ax, np.arange(23, 28))
    # The background on each side is the median of the nearby background profiles at each pressure
    nearby = select_nearby_background(eddy_profiles)
    for side in ('north', 'south'):
        grids = [interpolate_to_pressure_grid(index, valid[index]) for index in np.flatnonzero(nearby & profiles['side'].eq(side).to_numpy()).tolist()]
        median_salinity = cast(pd.Series, pd.DataFrame([salinity for salinity, _, _ in grids]).median()).to_numpy()
        median_temperature = cast(pd.Series, pd.DataFrame([temperature for _, temperature, _ in grids]).median()).to_numpy()
        ax.plot(median_salinity, median_temperature, color=side_colors[side], linewidth=4, alpha=0.45, solid_capstyle='round', zorder=2)
    day_norm = add_day_colorbar(fig, ax, eddy_profiles['days_from_crossing'])
    for profile_index, profile_days in zip(eddy_profiles['profile'], eddy_profiles['days_from_crossing']):
        color = time_cmap(day_norm(profile_days))
        keep = valid[profile_index]
        ax.plot(absolute_salinity[profile_index, keep], conservative_temperature[profile_index, keep], color=color, linewidth=1.1, zorder=3)
        salinity, temperature, _ = interpolate_to_pressure_grid(profile_index, valid[profile_index])
        ax.plot(salinity[marked_pressures], temperature[marked_pressures], linestyle='none', marker='o', markersize=3, color=color, markeredgecolor='white', markeredgewidth=0.4, zorder=4)
    # Panel (a) has room right of its curves to name the marked pressures on its earliest profile
    if letter == 'a':
        salinity, temperature, _ = interpolate_to_pressure_grid(eddy_profiles['profile'].iloc[0], valid[eddy_profiles['profile'].iloc[0]])
        for marked_salinity, marked_temperature, marked_pressure in zip(salinity[marked_pressures], temperature[marked_pressures], pressure_grid[marked_pressures]):
            ax.annotate(f'{marked_pressure} dbar', (marked_salinity, marked_temperature), xytext=(8, 0), textcoords='offset points', va='center', color='#555555')
    n_floats = eddy_profiles['float_id'].nunique()
    ax.set_title(rf'$\bf{{({letter})}}$ {polarity.capitalize()} {track_id} ({len(eddy_profiles)} profiles, {n_floats} float{"s" if n_floats > 1 else ""})', loc='left', color=polarity_colors[polarity])
# The lowest drawn panel of each column carries the x tick labels and label
used_axes = list(axes.flat[:len(case_eddies)])
for column in range(2):
    column_axes = [ax for ax in axes[:, column] if ax in used_axes]
    cast(Axes, column_axes[-1]).tick_params(labelbottom=True)
    cast(Axes, column_axes[-1]).set_xlabel('Absolute salinity (g kg$^{-1}$)')
for ax in axes[:, 0]:
    cast(Axes, ax).set_ylabel('Conservative temperature (°C)')
handles = [
    Line2D([], [], color=side_colors['north'], linewidth=4, alpha=0.45, label='North-side background (median)'),
    Line2D([], [], color=side_colors['south'], linewidth=4, alpha=0.45, label='South-side background (median)'),
    Line2D([], [], color='#555555', marker='o', markersize=3, markeredgecolor='white', markeredgewidth=0.4, linestyle='none', label='100, 300, and 600 dbar'),
]
if spare_cell:
    legend_ax = cast(Axes, axes.flat[len(case_eddies)])
    legend_ax.axis('off')
    legend_ax.legend(handles=handles, loc='center', handlelength=1.8)
else:
    fig.legend(handles=handles, loc='outside lower center', ncol=3, handlelength=1.8, columnspacing=1.5)
plt.show()

## Fixed depth and fixed density

An eddy can turn warmer at 300 m because warmer water mixed into it, or because its warm layers moved down (heave). Heave moves water up or down without changing it, so temperature and salinity on a density surface stay the same. A change on a density surface therefore means the water itself changed, though not what changed it. The first figure shows each case eddy against depth, the second its salinity on the three density surfaces of the water-mass table beside the depth of those surfaces. On a density surface temperature follows from salinity, so salinity alone is shown.

In [ ]:
# Each case eddy's profiles against depth, temperature as the anomaly from the seasonal baseline, colored by days from the crossing as in the T/S panels; the thick lines are the median of the nearby background on each side at each pressure
fig, axes = cast(tuple[Figure, np.ndarray], plt.subplots(len(case_eddies), 2, figsize=(6.69, 2.3 * len(case_eddies) + 0.4), layout='constrained', sharex='col', sharey=True))
for row, (polarity, track_id) in enumerate(case_eddies.itertuples(index=False)):
    temperature_ax, salinity_ax = cast(tuple[Axes, Axes], axes[row])
    eddy_profiles = interior_temperature.loc[interior_temperature['polarity'].eq(polarity) & interior_temperature['track_id'].eq(track_id)].sort_values('date')
    # Temperature also comes from temperature-only profiles, in the eddy and in the background; salinity only from profiles with salinity
    nearby = select_nearby_background(interior.loc[interior['polarity'].eq(polarity) & interior['track_id'].eq(track_id)])
    nearby_temperature = select_nearby_background(eddy_profiles, 'background_temperature')
    grid_depth = -gsw.z_from_p(pressure_grid, eddy_profiles['lat'].mean())
    for side in ('north', 'south'):
        on_side = profiles['side'].eq(side).to_numpy()
        salinity_grids = [interpolate_to_pressure_grid(index, valid[index]) for index in np.flatnonzero(nearby & on_side).tolist()]
        temperature_grids = [interpolate_to_pressure_grid(index, valid_temperature[index]) for index in np.flatnonzero(nearby_temperature & on_side).tolist()]
        median_salinity = cast(pd.Series, pd.DataFrame([salinity for salinity, _, _ in salinity_grids]).median()).to_numpy()
        median_anomaly = cast(pd.Series, pd.DataFrame([anomaly for _, _, anomaly in temperature_grids]).median()).to_numpy()
        temperature_ax.plot(median_anomaly, grid_depth, color=side_colors[side], linewidth=4, alpha=0.45, solid_capstyle='round', zorder=2)
        salinity_ax.plot(median_salinity, grid_depth, color=side_colors[side], linewidth=4, alpha=0.45, solid_capstyle='round', zorder=2)
    day_norm = add_day_colorbar(fig, salinity_ax, eddy_profiles['days_from_crossing'])
    for profile_index, profile_days in zip(eddy_profiles['profile'], eddy_profiles['days_from_crossing']):
        color = time_cmap(day_norm(profile_days))
        keep = valid_temperature[profile_index]
        levels = np.argsort(depth[profile_index, keep])
        temperature_ax.plot(temperature_anomaly[profile_index, keep][levels], depth[profile_index, keep][levels], color=color, linewidth=1.1, zorder=3)
        keep = valid[profile_index]
        levels = np.argsort(depth[profile_index, keep])
        salinity_ax.plot(absolute_salinity[profile_index, keep][levels], depth[profile_index, keep][levels], color=color, linewidth=1.1, zorder=3)
    temperature_ax.set_title(rf'$\bf{{({"abcdefghijklmnop"[2 * row]})}}$ {polarity.capitalize()} {track_id}, temperature anomaly', loc='left', color=polarity_colors[polarity])
    salinity_ax.set_title(rf'$\bf{{({"abcdefghijklmnop"[2 * row + 1]})}}$ {polarity.capitalize()} {track_id}, salinity', loc='left', color=polarity_colors[polarity])
    temperature_ax.set_ylabel('Depth (m)')
    for ax in (temperature_ax, salinity_ax):
        ax.grid(color='#e5e5e5', linewidth=0.5)
        ax.set_axisbelow(True)
axes[0, 0].set_ylim(1000, 0)
axes[-1, 0].set_xlabel('Temperature anomaly (°C)')
axes[-1, 1].set_xlabel('Absolute salinity (g kg$^{-1}$)')
handles = [Line2D([], [], color=side_colors[side], linewidth=4, alpha=0.45, label=f'{side.capitalize()}-side background (median)') for side in ('north', 'south')]
fig.legend(handles=handles, loc='outside lower center', ncol=2, handlelength=1.8, columnspacing=1.5)
plt.show()

In [ ]:
# Salinity on each density surface of the water-mass table against days from the crossing, and the depth of those surfaces, one row per case eddy; dots carry the day colors of the figures above, and the thick lines are the median of the nearby background on each side on that surface
fig, axes = cast(tuple[Figure, np.ndarray], plt.subplots(len(case_eddies), len(DENSITY_SURFACES) + 1, figsize=(6.69, 1.9 * len(case_eddies) + 0.4), layout='constrained', sharex='row', sharey='col'))
letters = iter('abcdefghijklmnopqrstuvwxyz')
for row, (polarity, track_id) in enumerate(case_eddies.itertuples(index=False)):
    eddy_profiles = interior.loc[interior['polarity'].eq(polarity) & interior['track_id'].eq(track_id)].sort_values('date')
    nearby = select_nearby_background(eddy_profiles)
    days = eddy_profiles['days_from_crossing'].to_numpy()
    day_colors = time_cmap(Normalize(days.min(), days.max())(days))
    eddy_surfaces = on_surfaces[eddy_profiles['profile'].to_numpy()]
    surface_depths = -gsw.z_from_p(eddy_surfaces[:, :, 2], eddy_profiles['lat'].to_numpy()[:, None])
    depth_ax = cast(Axes, axes[row, -1])
    for column, surface in enumerate(DENSITY_SURFACES):
        salinity_ax = cast(Axes, axes[row, column])
        for side in ('north', 'south'):
            salinity_ax.axhline(float(np.nanmedian(on_surfaces[nearby & profiles['side'].eq(side).to_numpy(), column, 1])), color=side_colors[side], linewidth=4, alpha=0.45, zorder=1)
        for ax, values in ((salinity_ax, eddy_surfaces[:, column, 1]), (depth_ax, surface_depths[:, column])):
            held = np.isfinite(values)
            ax.plot(days[held], values[held], color='#999999', linewidth=0.6, zorder=2)
            ax.scatter(days[held], values[held], c=day_colors[held], s=14, edgecolors='white', linewidths=0.4, zorder=3)
        # Name each surface beside its last point in the depth panel
        held = np.flatnonzero(np.isfinite(surface_depths[:, column]))
        if len(held) > 0:
            depth_ax.annotate(f'{surface:g}', (days[held[-1]], surface_depths[held[-1], column]), xytext=(4, 0), textcoords='offset points', va='center', fontsize=7, color='#555555')
    titles = [rf'$\sigma_0$ = {surface:g}' for surface in DENSITY_SURFACES] + ['Depth of surfaces'] if row == 0 else [''] * (len(DENSITY_SURFACES) + 1)
    for ax, title in zip(axes[row], titles):
        ax = cast(Axes, ax)
        ax.set_title(rf'$\bf{{({next(letters)})}}$ {title}', loc='left')
        if days.min() < 0 < days.max():
            ax.axvline(0, color='black', linewidth=0.8, zorder=1)
        ax.grid(color='#e5e5e5', linewidth=0.5)
        ax.set_axisbelow(True)
        ax.xaxis.set_major_locator(MaxNLocator(nbins=4, integer=True))
    cast(Axes, axes[row, 0]).set_ylabel(f'{polarity.capitalize()} {track_id}\nSalinity (g kg$^{{-1}}$)', color=polarity_colors[polarity])
    depth_ax.set_ylabel('Depth (m)')
cast(Axes, axes[0, -1]).set_ylim(1000, 0)
for ax in axes[-1]:
    cast(Axes, ax).set_xlabel('Days from crossing')
handles = [
    *[Line2D([], [], color=side_colors[side], linewidth=4, alpha=0.45, label=f'{side.capitalize()}-side background (median)') for side in ('north', 'south')],
    Line2D([], [], color='black', linewidth=0.8, label='Crossing'),
]
fig.legend(handles=handles, loc='outside lower center', ncol=3, handlelength=1.8, columnspacing=1.5)
plt.show()

## Mean over 0 to 1000 dbar

In [ ]:
# One dot per profile: salinity and temperature anomaly averaged over 0 to 1000 dbar, each level weighted by the pressure it spans, for profiles that reach from the top 10 dbar to 950 dbar; this mean tells the two sides of the Gulf Stream apart far better than the surface value, which shelf water and the seasonal cycle left in north-side anomalies scatter
# Each map is a fixed box of degrees around the eddy's contours on its profile days, moved to stay inside the downloaded Copernicus region, wide enough in latitude to reach the axis from the farthest profile
MAP_LON_DEG = 12
MAP_LAT_DEG = 8
plankton_files = sorted((DATA_DIR / 'bronze/plankton').glob('plankton_*.nc'))
chl_norm = LogNorm(0.05, 2)  # pyright: ignore[reportCallIssue]
row_inches = 1.5
# One row per case eddy, with 0.45 in above the rows and 1.4 in below them for the colorbar and the legend
fig_height = 0.45 + row_inches * len(case_eddies) + 1.4
letters = 'abcdefghijklmnopqrstuvwxyz'
map_axes: list[GeoAxes] = []
fig = plt.figure(figsize=(6.69, fig_height))
for row, (polarity, track_id) in enumerate(case_eddies.itertuples(index=False)):
    bottom = 1 - (0.45 + (row + 1) * row_inches) / fig_height
    ax = cast(Axes, fig.add_axes((0.1, bottom, 0.4, (row_inches - 0.42) / fig_height)))
    eddy_profiles = interior.loc[interior['polarity'].eq(polarity) & interior['track_id'].eq(track_id)].sort_values('date').join(profiles[['column_salinity', 'column_temperature_anomaly']], on='profile')
    ax.set_xlim(34.75, 36.75)
    ax.set_ylim(-10, 5)
    nearby = select_nearby_background(eddy_profiles)
    for side in ('north', 'south'):
        chosen = profiles.loc[nearby & profiles['side'].eq(side).to_numpy()]
        ax.scatter(chosen['column_salinity'], chosen['column_temperature_anomaly'], s=8, color=side_colors[side], alpha=0.35, linewidths=0, zorder=2)
    day_norm = add_day_colorbar(fig, ax, eddy_profiles['days_from_crossing'])
    day_colors = time_cmap(day_norm(eddy_profiles['days_from_crossing'].to_numpy()))
    ax.scatter(eddy_profiles['column_salinity'], eddy_profiles['column_temperature_anomaly'], s=30, color=day_colors, edgecolors='black', linewidths=0.5, zorder=3)
    n_floats = eddy_profiles['float_id'].nunique()
    ax.set_title(rf'$\bf{{({letters[2 * row]})}}$ {polarity.capitalize()} {track_id} ({len(eddy_profiles)} profiles, {n_floats} float{"s" if n_floats > 1 else ""})', loc='left', color=polarity_colors[polarity])
    ax.set_ylabel('Temperature anomaly,\n0 to 1000 dbar mean (°C)')
    if row < len(case_eddies) - 1:
        ax.tick_params(labelbottom=False)
    else:
        ax.set_xlabel('Absolute salinity, 0 to 1000 dbar mean (g kg$^{-1}$)')

    # Copernicus CHL averaged over every day from the first to the last profile, so the clouds of single days fill in
    first_day, last_day = eddy_profiles['date'].min(), eddy_profiles['date'].max()
    daily_chl = [chl for chl in (xr.open_dataset(path)['CHL'].sel(time=slice(first_day, last_day)) for path in plankton_files) if chl.sizes['time']]
    mean_chl = xr.concat(daily_chl, 'time').mean('time')
    contour_observations = observations.loc[observations['polarity'].eq(polarity) & observations['track_id'].eq(track_id) & observations['date'].isin(eddy_profiles['date'])]
    contour_lons = np.concatenate(contour_observations['contour_lon'].to_list())
    contour_lats = np.concatenate(contour_observations['contour_lat'].to_list())
    west = float(np.clip((contour_lons.min() + contour_lons.max() - MAP_LON_DEG) / 2, -81, -56 - MAP_LON_DEG))
    south = float(np.clip((contour_lats.min() + contour_lats.max() - MAP_LAT_DEG) / 2, 29, 44 - MAP_LAT_DEG))
    map_ax = cast(GeoAxes, fig.add_axes((0.6, bottom, 0.3, (row_inches - 0.42) / fig_height), projection=ccrs.PlateCarree()))  # pyright: ignore[reportArgumentType]
    map_axes.append(map_ax)
    map_ax.set_extent([west, west + MAP_LON_DEG, south, south + MAP_LAT_DEG], crs=ccrs.PlateCarree())
    map_ax.pcolormesh(mean_chl['longitude'], mean_chl['latitude'], mean_chl, cmap='PuBu', norm=chl_norm, shading='nearest', rasterized=True, transform=ccrs.PlateCarree(), zorder=1)
    map_ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='#e9e9e9', zorder=2)
    map_ax.coastlines(resolution='50m', color='#666666', linewidth=0.5, zorder=3)
    gridlines = map_ax.gridlines(draw_labels=True, linewidth=0.5, color='#bbbbbb', alpha=0.6, xlocs=range(-80, -51, 4), ylocs=range(28, 46, 2), zorder=3)
    gridlines.top_labels = False
    gridlines.right_labels = False
    gridlines.xlabel_style = {'size': 8}
    gridlines.ylabel_style = {'size': 8}
    # The axis and the contour of each profile day share that day's color with the dots on the left
    ordered = contour_observations.sort_values('date')
    for day, contour_lon, contour_lat, color in zip(ordered['date'], ordered['contour_lon'], ordered['contour_lat'], [day_colors[eddy_profiles['date'].eq(day).to_numpy()][0] for day in ordered['date']]):
        centerline = centerlines[day.date()]
        map_ax.plot(centerline.lon, centerline.lat, color=color, linewidth=0.8, transform=ccrs.PlateCarree(), zorder=4)
        map_ax.plot(np.append(contour_lon, contour_lon[0]), np.append(contour_lat, contour_lat[0]), color=color, linewidth=1.3, path_effects=[patheffects.withStroke(linewidth=2.3, foreground='black')], transform=ccrs.PlateCarree(), zorder=5)
    map_ax.scatter(eddy_profiles['lon'], eddy_profiles['lat'], s=6, color='white', edgecolors='black', linewidths=0.4, transform=ccrs.PlateCarree(), zorder=6)
    map_ax.set_title(rf'$\bf{{({letters[2 * row + 1]})}}$ Mean CHL, {first_day:%-d %b %Y} to {last_day:%-d %b %Y}', loc='left')
# The colorbar spans the drawn width of the maps, which their equal-degree aspect narrows inside the axes box
map_axes[-1].apply_aspect()
map_position = map_axes[-1].get_position()
chl_colorbar = fig.colorbar(ScalarMappable(cmap='PuBu', norm=chl_norm), cax=fig.add_axes((map_position.x0, 0.65 / fig_height, map_position.width, 0.11 / fig_height)), orientation='horizontal')
chl_colorbar.outline.set_linewidth(0.5)  # pyright: ignore[reportCallIssue]
chl_colorbar.set_ticks([0.05, 0.1, 0.2, 0.5, 1, 2], labels=['0.05', '0.1', '0.2', '0.5', '1', '2'])
chl_colorbar.set_label('Copernicus CHL (mg m$^{-3}$)', labelpad=1)
handles = [
    Line2D([], [], color=side_colors['north'], marker='o', markersize=4, alpha=0.5, linestyle='none', label='North-side background'),
    Line2D([], [], color=side_colors['south'], marker='o', markersize=4, alpha=0.5, linestyle='none', label='South-side background'),
    Line2D([], [], color='#999999', marker='o', markersize=5, markeredgecolor='black', markeredgewidth=0.5, linestyle='none', label='Eddy profile'),
    Line2D([], [], color='#6abf3a', linewidth=1.3, path_effects=[patheffects.withStroke(linewidth=2.3, foreground='black')], label='Speed contour on a profile day'),
    Line2D([], [], color='#6abf3a', linewidth=0.8, label='Gulf Stream axis on a profile day'),
    Line2D([], [], color='white', marker='o', markersize=3, markeredgecolor='black', markeredgewidth=0.4, linestyle='none', label='Profile position'),
]
fig.legend(handles=handles, loc='lower left', bbox_to_anchor=(0.02, 0.15 / fig_height), ncol=2, handlelength=1.6, columnspacing=1)
plt.show()

## Satellite SST and SSS

In [ ]:
# Satellite matchups: each MODIS-Aqua 8-day SST composite, as the anomaly from the regional seasonal baseline on its middle day, and the SMAP 8-day SSS map centered on the same day, inside the eddy contour of the composite's middle day, and in the background on each side of that day's axis within REFERENCE_LAT_DEG of the eddy
MIN_SST_PIXELS = 10
MIN_SSS_CELLS = 3
case_observations = observations.merge(case_eddies, on=identity_columns)
satellite_rows = []
for sst_path in sorted((DATA_DIR / 'bronze/sst').glob('*.nc')):
    start, end = (dt.datetime.strptime(stamp, '%Y%m%d').date() for stamp in re.search(r'(\d{8})_(\d{8})', sst_path.name).groups())  # pyright: ignore[reportOptionalMemberAccess]
    midpoint_day = start + dt.timedelta(days=(end - start).days // 2)
    midpoint = pd.Timestamp(midpoint_day)
    day_eddies = case_observations.loc[case_observations['date'].eq(midpoint)]
    sss_path = DATA_DIR / f'bronze/sss/SMAP_L3_SSS_{midpoint_day:%Y%m%d}_8DAYS_V5_subsetted.nc4'
    if day_eddies.empty or not sss_path.exists():
        continue
    with xr.open_dataset(sst_path) as sst_file, xr.open_dataset(sss_path) as sss_file:
        # Quality 2 is cloud-contaminated cold pixels that would fake a cold eddy
        fields = {
            'sst': (sst_file['sst'].where(sst_file['qual_sst'] <= 1).to_numpy() - float(estimate_seasonal_baseline(midpoint.dayofyear, np.array(0.0))), sst_file['lon'].to_numpy(), sst_file['lat'].to_numpy(), MIN_SST_PIXELS),
            'sss': (sss_file['smap_sss'].to_numpy(), sss_file['longitude'].to_numpy(), sss_file['latitude'].to_numpy(), MIN_SSS_CELLS),
        }
    centerline = centerlines[midpoint_day]
    swot_path = find_nearest_swot_file(swot_files, midpoint_day)
    day_contours = observations_by_date[midpoint]
    for polarity, track_id, contour_lon, contour_lat, center_lat in zip(day_eddies['polarity'], day_eddies['track_id'], day_eddies['contour_lon'], day_eddies['contour_lat'], day_eddies['center_lat']):
        row = {'polarity': polarity, 'track_id': track_id, 'date': midpoint}
        for name, (field, lon, lat, min_count) in fields.items():
            inside = mask_pixels_inside_contour(lon, lat, contour_lon, contour_lat)
            values = field[inside]
            n_valid = int(np.isfinite(values).sum())
            row[name] = np.nanmean(values) if n_valid >= max(min_count, inside.sum() / 2) else np.nan
            # Background: calm water outside every eddy contour of the day, split by the side of the axis
            band = np.abs(lat - center_lat) <= REFERENCE_LAT_DEG
            band_lon, band_lat = np.meshgrid(lon, lat[band])
            nearby_contours = day_contours.loc[day_contours['center_lat'].sub(center_lat).abs().lt(REFERENCE_LAT_DEG + 3)]
            outside = ~is_in_any_contour(list(zip(nearby_contours['contour_lon'], nearby_contours['contour_lat'])), band_lon.ravel(), band_lat.ravel()).reshape(band_lon.shape)
            calm = compute_calm_mask_on_pace(swot_path, lon, lat[band]) if swot_path is not None else np.zeros(band_lon.shape, dtype=bool)
            north = compute_signed_distance_grid_km(centerline.lon, centerline.lat, lon, lat[band]) > 0
            for side, on_side in (('north', north), ('south', ~north)):
                # A side can hold no calm, non-eddy water in the band, and then has no background
                side_values = field[band][outside & calm & on_side]
                row[f'{name}_{side}'] = np.nanmean(side_values) if np.isfinite(side_values).any() else np.nan
        satellite_rows.append(row)
satellite = pd.DataFrame(satellite_rows)
satellite['days_from_crossing'] = [(date - crossing_dates[polarity, track_id]).days for date, polarity, track_id in zip(satellite['date'], satellite['polarity'], satellite['track_id'])]
satellite['matched'] = satellite['sst'].notna() & satellite['sss'].notna()
display(cast(pd.DataFrame, satellite.assign(
    before=satellite['matched'] & satellite['days_from_crossing'].lt(0), after=satellite['matched'] & satellite['days_from_crossing'].ge(0),
).groupby(identity_columns).agg(
    composites=('date', 'size'), with_sst=('sst', 'count'), with_sss=('sss', 'count'), both_before_crossing=('before', 'sum'), both_after_crossing=('after', 'sum'),
)))

# The same grid as the profile figure: two columns, with the legend in the spare cell when the eddy count is odd
fig, axes = cast(tuple[Figure, np.ndarray], plt.subplots(n_rows, 2, figsize=(6.69, 2.8 * n_rows + (0 if spare_cell else 0.5)), layout='constrained', sharex=True, sharey=True, squeeze=False))
for letter, ax, (polarity, track_id) in zip('abcdefghijklmnopqrstuvwxyz', axes.flat, case_eddies.itertuples(index=False)):
    ax = cast(Axes, ax)
    eddy_composites = satellite.loc[satellite['polarity'].eq(polarity) & satellite['track_id'].eq(track_id)]
    for side in ('north', 'south'):
        ax.scatter(eddy_composites[f'sss_{side}'], eddy_composites[f'sst_{side}'], s=12, color=side_colors[side], alpha=0.45, linewidths=0, zorder=2)
    matched = eddy_composites.loc[eddy_composites['matched']]
    day_norm = add_day_colorbar(fig, ax, eddy_composites['days_from_crossing'])
    ax.scatter(matched['sss'], matched['sst'], s=30, color=time_cmap(day_norm(matched['days_from_crossing'].to_numpy())), edgecolors='black', linewidths=0.5, zorder=3)
    ax.set_title(rf'$\bf{{({letter})}}$ {polarity.capitalize()} {track_id} ({len(matched)} of {len(eddy_composites)} composites)', loc='left', color=polarity_colors[polarity])
used_axes = list(axes.flat[:len(case_eddies)])
for column in range(2):
    column_axes = [ax for ax in axes[:, column] if ax in used_axes]
    cast(Axes, column_axes[-1]).tick_params(labelbottom=True)
    cast(Axes, column_axes[-1]).set_xlabel('SMAP sea surface salinity (psu)')
for ax in axes[:, 0]:
    cast(Axes, ax).set_ylabel('MODIS SST anomaly (°C)')
handles = [
    Line2D([], [], color=side_colors['north'], marker='o', markersize=4, alpha=0.6, linestyle='none', label='North-side background'),
    Line2D([], [], color=side_colors['south'], marker='o', markersize=4, alpha=0.6, linestyle='none', label='South-side background'),
    Line2D([], [], color='#999999', marker='o', markersize=5, markeredgecolor='black', markeredgewidth=0.5, linestyle='none', label='Eddy interior'),
]
if spare_cell:
    legend_ax = cast(Axes, axes.flat[len(case_eddies)])
    legend_ax.axis('off')
    legend_ax.legend(handles=handles, loc='center', handlelength=1.2, title='One dot per 8-day composite\nwith both SST and SSS', alignment='left')
else:
    fig.legend(handles=handles, loc='outside lower center', ncol=3, handlelength=1.2, columnspacing=1.5, title='One dot per 8-day composite with both SST and SSS')
plt.show()